# Logo Composition Analysis (OpenCV)

Analyses the logos collected by the logo scraper (`output/logos_metadata.csv` + `assets/logos/<site>/...`)
to extract measurable patterns that inform the creation of the Chapadmalal platform logo.

**Pipeline**
1. Inventory and validation of the scraped assets
2. Loading (raster + SVG), background detection and foreground mask extraction
3. Feature extraction with OpenCV: geometry, composition (symbol / wordmark), color, shape, stroke, symmetry, scalability, complexity
4. SVG structure analysis (vector logos)
5. Favicon / app icon relationship with the main logo
6. Derived classifications, archetype clustering and report-ready tables
7. Publication-quality charts, contact sheets and validation overlays

Nothing is hardcoded about findings: every statistic is computed at run time. Composition labels are
**heuristics**; the overlay sheets and the optional manual-label file exist to validate them.

In [1]:
import hashlib
import io
import math
import re
import warnings
from collections import Counter
from itertools import combinations
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

try:
    from PIL import Image
    PIL_OK = True
except Exception:
    PIL_OK = False

try:
    import cairosvg
    SVG_RENDER_OK = True
except Exception:  # ImportError or missing libcairo
    SVG_RENDER_OK = False

print("OpenCV:", cv2.__version__, "| Pillow:", PIL_OK, "| SVG rendering (cairosvg):", SVG_RENDER_OK)

OpenCV: 5.1.0 | Pillow: True | SVG rendering (cairosvg): False


## 0. Configuration

In [2]:
PROJECT_ROOT = Path(".")                                   # folder containing output/ and assets/
METADATA_CSV = PROJECT_ROOT / "output" / "logos_metadata.csv"
LOGOS_DIR = PROJECT_ROOT / "assets" / "logos"
SEGMENTS_CSV = PROJECT_ROOT / "analysis" / "quality" / "market_classification_template.csv"   # optional
OUT = PROJECT_ROOT / "analysis" / "logos"                  # all outputs live here
MANUAL_LABELS_CSV = OUT / "manual_labels.csv"              # optional: columns domain_key, composition_manual

DEDUP_BY_BRAND = False     # False = every site counts (default). True = one logo per brand (all tripadvisor_* domains count once)
EXCLUDE_PHOTOGRAPHIC = False  # False = photographic / complex logos are only flagged; True = they are excluded from the statistics
FALLBACK_TO_VARIANT = True # if the primary logo is unusable, use the best successful variant for that site
SEED = 42

# --- Preprocessing ---------------------------------------------------------------------------------
TARGET_AREA = 256 * 256 * 2   # normalised analysis area (pixels) of the cropped logo
MAX_SIDE = 1024               # maximum side of the normalised crop
SVG_RASTER_WIDTH = 1024       # rasterisation width for SVG files
ALPHA_THRESHOLD = 16          # alpha > threshold counts as foreground for transparent images
FG_DELTAE = 10.0              # Lab distance to background that counts as foreground (opaque images)
BG_UNIFORMITY = 0.90          # share of border pixels that must match the border median to call the background solid
MIN_COMPONENT_AREA_RATIO = 0.0002   # connected components smaller than this share of the crop are treated as specks

# --- Composition heuristics (all thresholds are explicit and tunable) --------------------------------
GROUP_GAP_RATIO = 0.12        # dilation radius / crop height used to merge letters into groups
TEXT_MIN_COMPONENTS = 3       # minimum letters (components) for a text-like group
TEXT_MIN_ASPECT = 1.8         # minimum width/height of a text-like group
WORDMARK_ASPECT = 3.0         # a connected (script) group at least this wide is also text-like
MIN_GROUP_SHARE = 0.02        # groups below this share of foreground area are ignored (dots, TM marks)
CONTAINER_BBOX_RATIO = 0.85   # outer contour must span this share of the logo bounding box
CONTAINER_INNER_SHARE = 0.05  # share of foreground that must lie inside the outer contour as nested shapes

# --- Color --------------------------------------------------------------------------------------------
COLOR_K = 6                   # k-means clusters before merging
MERGE_DELTAE = 14.0           # clusters closer than this (CIE76 in Lab) are merged
MIN_COLOR_SHARE = 0.02        # palette colors below this foreground share are dropped
MAX_COLOR_PIXELS = 20000
GRADIENT_SPREAD = 6.0         # mean deltaE of core pixels to their palette color above which a gradient is flagged
PHOTO_UNIQUE_COLORS = 400     # unique 4-bit colors above which a logo is flagged as photographic / complex
ACHROMATIC_CHROMA = 12.0      # maximum Lab chroma for all palette colors to call a logo achromatic
TINT_MAX_SHARE = 0.15         # minor colors up to this share may be anti-aliasing tints of a larger color
TINT_DELTAE = 10.0            # a minor color within this Lab distance of the line (main color -> background) is merged into the main color

# --- Other ---------------------------------------------------------------------------------------------
SMALL_HEIGHTS = [48, 24, 16]  # pixel heights used for the scalability test
SYMMETRY_HIGH = 0.90          # IoU above which a logo is reported as highly symmetric
ASPECT_BINS = [0, 0.8, 1.25, 2.0, 3.5, np.inf]
ASPECT_LABELS = ["Vertical (< 0.8)", "Near-square (0.8-1.25)", "Landscape (1.25-2)", "Wide (2-3.5)", "Very wide (> 3.5)"]
CLUSTER_K_RANGE = range(2, 7)
DPI = 300

cv2.setRNGSeed(SEED)
RNG = np.random.default_rng(SEED)

## 1. Output registry and plotting helpers

In [3]:
TABLES, CHARTS, SHEETS_DIR, PROCESSED = OUT / "tables", OUT / "charts", OUT / "contact_sheets", OUT / "processed"
for folder in (OUT, TABLES, CHARTS, SHEETS_DIR, PROCESSED):
    folder.mkdir(parents=True, exist_ok=True)

MANIFEST, VALIDATION, EXCEL_TABLES, R = [], [], {}, {}

DIM = ["Inventory", "Geometry", "Composition", "Color", "Shape and stroke", "Scalability and complexity",
       "Icons", "SVG structure", "Variants", "Archetypes", "Validation"]


def validate(dim, item, status, detail="", n=np.nan):
    VALIDATION.append({"Dimension": dim, "Item": item, "Status": status, "Detail": detail, "n": n})


def pct(part, base):
    return round(100.0 * part / base, 2) if base else np.nan


def save_table(df, name, dim, description):
    if df is None or len(df) == 0:
        validate(dim, name, "NOT_GENERATED", f"No rows: {description}")
        return None
    df = df.copy()
    for c in df.select_dtypes(include=["float"]).columns:
        df[c] = df[c].round(3 if "ratio" in c.lower() or "iou" in c.lower() else 2)
    df.to_csv(TABLES / name, index=False, encoding="utf-8-sig")
    MANIFEST.append({"Dimension": dim, "Type": "table", "Filename": f"tables/{name}", "Rows": len(df), "Description": description})
    EXCEL_TABLES.setdefault(dim, []).append((name.replace(".csv", ""), description, df))
    return df


def dist_table(series, label, dim_note=None):
    s = series.dropna()
    t = s.value_counts().rename_axis(label).reset_index(name="Logos (n)")
    t["Logos (%)"] = [pct(v, len(s)) for v in t["Logos (n)"]]
    t["Base (n)"] = len(s)
    return t


def describe_cols(df, cols, labels=None):
    rows = []
    for c in cols:
        if c not in df.columns:
            continue
        v = pd.to_numeric(df[c], errors="coerce").dropna()
        row = {"Metric": (labels or {}).get(c, c), "Valid (n)": len(v), "Missing (n)": len(df) - len(v)}
        if len(v):
            row.update({"Mean": v.mean(), "Median": v.median(), "Std. dev.": v.std(), "Min": v.min(),
                        "P25": v.quantile(.25), "P75": v.quantile(.75), "P90": v.quantile(.9), "Max": v.max()})
        rows.append(row)
    return pd.DataFrame(rows)


def crosstab_pct(df, row, col, row_label, col_label):
    d = df[[row, col]].dropna()
    if d.empty:
        return pd.DataFrame()
    ct = pd.crosstab(d[row], d[col])
    out = ct.div(ct.sum(axis=1), axis=0) * 100
    long = out.stack().rename("Row share (%)").reset_index()
    long.columns = [row_label, col_label, "Row share (%)"]
    counts = ct.stack().rename("Logos (n)").reset_index()
    counts.columns = [row_label, col_label, "Logos (n)"]
    long = long.merge(counts, on=[row_label, col_label])
    long["Row base (n)"] = long[row_label].map(ct.sum(axis=1))
    return long[long["Logos (n)"] > 0]


C_PRIMARY, C_SECONDARY, C_ACCENT, C_GREY = "#1F4E79", "#2E86AB", "#F18F01", "#8D99AE"
CAT = ["#1F4E79", "#2E86AB", "#F18F01", "#6A994E", "#BC4749", "#7B6D8D", "#8D99AE", "#E9C46A", "#264653", "#A8DADC"]
FAMILY_COLORS = {"Neutral": "#9E9E9E", "Red": "#D32F2F", "Orange": "#F57C00", "Yellow": "#FBC02D", "Green": "#388E3C",
                 "Cyan": "#00ACC1", "Blue": "#1976D2", "Purple": "#7B1FA2", "Pink": "#E91E63"}
plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 10, "axes.titlesize": 11, "axes.titleweight": "bold",
                     "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.alpha": .25,
                     "axes.axisbelow": True, "legend.frameon": False, "legend.fontsize": 9, "savefig.dpi": DPI})


def new_fig(rows, cols, title, figsize=None):
    fig, axes = plt.subplots(rows, cols, figsize=figsize or (7.5 * cols, 5.2 * rows))
    fig.suptitle(title, fontsize=15, fontweight="bold", y=0.995)
    return fig, np.atleast_1d(axes).ravel()


def save_fig(fig, name, dim, description, n_label):
    fig.text(0.01, 0.005, f"Source: logos scraped from top tourism websites. {n_label}", fontsize=8, color="#555", va="bottom")
    fig.tight_layout(rect=[0, 0.025, 1, 0.965])
    fig.savefig(CHARTS / name, dpi=DPI, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    MANIFEST.append({"Dimension": dim, "Type": "chart", "Filename": f"charts/{name}", "Rows": np.nan, "Description": description})


def empty_panel(ax, title, reason="No data available"):
    ax.set_title(title, loc="left"); ax.text(.5, .5, reason, ha="center", va="center", color="#777", transform=ax.transAxes)
    ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)


def p_bar(ax, labels, values, title, xlabel, n=None, colors=None, fmt="{:.1f}%", xmax=None):
    labels, values = list(labels), list(values)
    t = f"{title} (n = {n})" if n is not None else title
    if not labels:
        return empty_panel(ax, t)
    y = np.arange(len(labels))
    bars = ax.barh(y, values, color=colors or C_PRIMARY, edgecolor="#333", linewidth=.4)
    ax.set_yticks(y); ax.set_yticklabels([str(l)[:34] for l in labels]); ax.invert_yaxis()
    top = xmax or (max(values) * 1.18 if max(values) > 0 else 1)
    ax.set_xlim(0, top); ax.set_xlabel(xlabel); ax.set_title(t, loc="left"); ax.grid(axis="y", visible=False)
    for b, v in zip(bars, values):
        ax.text(b.get_width() + top * .01, b.get_y() + b.get_height() / 2, fmt.format(v), va="center", fontsize=8)


def p_hist(ax, values, title, xlabel, bins=20, log_x=False, ylabel="Logos (n)"):

    v = pd.to_numeric(pd.Series(values), errors="coerce")

    # Remove NaN and infinite values
    v = v.replace([np.inf, -np.inf], np.nan).dropna()

    if log_x:
        v = v[v > 0]

    t = f"{title} (n = {len(v)})"

    if v.empty:
        return empty_panel(ax, t)

    if log_x and v.max() > v.min():
        bins = np.logspace(
            np.log10(v.min()),
            np.log10(v.max()),
            bins + 1
        )
        ax.set_xscale("log")

    ax.hist(v, bins=bins, color=C_PRIMARY, edgecolor="white")

    ax.axvline(
        v.median(),
        color=C_ACCENT,
        ls="--",
        lw=1.5,
        label=f"Median = {v.median():,.2f}"
    )

    ax.legend()
    ax.set_title(t, loc="left")
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)



def p_stack(ax, share, title, xlabel, n_text="", cmap=None):
    t = f"{title}{n_text}"
    if share is None or share.empty:
        return empty_panel(ax, t)
    left = np.zeros(len(share))
    for i, c in enumerate(share.columns):
        ax.barh(range(len(share)), share[c].values, left=left, color=(cmap or {}).get(c, CAT[i % len(CAT)]), edgecolor="white", label=str(c))
        left += share[c].values
    ax.set_yticks(range(len(share))); ax.set_yticklabels([str(i)[:30] for i in share.index]); ax.invert_yaxis()
    ax.set_xlim(0, 100); ax.set_xlabel(xlabel); ax.set_title(t, loc="left"); ax.grid(axis="y", visible=False)
    ax.legend(loc="upper center", bbox_to_anchor=(.5, -.18), ncol=min(len(share.columns), 3))

## 2. Load metadata, resolve files and validate the inventory

In [4]:
if not METADATA_CSV.exists():
    raise FileNotFoundError(f"Logo metadata not found: {METADATA_CSV}")

meta = pd.read_csv(METADATA_CSV, encoding="utf-8-sig", low_memory=False)
EXPECTED = ["domain", "asset_type", "format_detected", "source_url", "local_path", "selector", "alt_text", "width",
            "height", "confidence_score", "file_size_bytes", "sha256", "download_status"]
for c in EXPECTED:
    if c not in meta.columns:
        validate("Inventory", f"Column '{c}'", "MISSING", "Column absent from logos_metadata.csv; dependent metrics are skipped.")
        meta[c] = np.nan
    else:
        validate("Inventory", f"Column '{c}'", "OK", f"{int(meta[c].notna().sum())} of {len(meta)} rows contain data.", int(meta[c].notna().sum()))

meta = meta.reset_index(drop=True)
meta["asset_id"] = np.arange(len(meta))
meta["domain_key"] = meta["domain"].astype(str).str.lower().str.replace(r"^www\.", "", regex=True).str.split(":").str[0]


def brand_key(host):
    labels = host.split(".")
    if len(labels) >= 3 and labels[-2] in {"co", "com", "org", "net", "gov", "ne", "or", "ac"}:
        return labels[-3]
    return labels[-2] if len(labels) >= 2 else labels[0]


meta["brand_key"] = meta["domain_key"].apply(brand_key)


def resolve_path(local_path):
    if not isinstance(local_path, str) or not local_path.strip():
        return None
    p = Path(local_path)
    candidates = [p, PROJECT_ROOT / p]
    if "logos" in p.parts:
        candidates.append(LOGOS_DIR / Path(*p.parts[p.parts.index("logos") + 1:]))
    for c in candidates:
        if c.exists():
            return c
    return None


meta["file_path"] = meta["local_path"].apply(resolve_path)
meta["file_exists"] = meta["file_path"].notna()
meta["is_success"] = meta["download_status"].astype(str).eq("success")

# Role follows the scraper's counters: the first 'logo' row of a domain is the primary logo (failures also count).
is_logo = meta["asset_type"].eq("logo")
meta["logo_rank"] = np.nan
meta.loc[is_logo, "logo_rank"] = meta[is_logo].groupby("domain_key").cumcount() + 1
meta["role"] = np.select([is_logo & (meta["logo_rank"] == 1), is_logo], ["primary", "variant"], default=meta["asset_type"])

R["meta"] = meta
broken = meta[meta["is_success"] & ~meta["file_exists"]]
if len(broken):
    validate("Inventory", "Successful downloads whose file was not found", "PARTIAL",
             f"{len(broken)} rows report success but the file could not be resolved (check PROJECT_ROOT).", len(broken))
if LOGOS_DIR.exists():
    on_disk = {p.resolve() for p in LOGOS_DIR.rglob("*") if p.is_file()}
    known = {p.resolve() for p in meta["file_path"].dropna()}
    orphans = on_disk - known
    if orphans:
        validate("Inventory", "Files on disk without metadata row", "INFO", f"{len(orphans)} files are not referenced in logos_metadata.csv.", len(orphans))
else:
    validate("Inventory", "Logos directory", "MISSING", f"{LOGOS_DIR} not found.")

n_sites = meta["domain_key"].nunique()
print(f"Rows: {len(meta)} | sites: {n_sites} | brands: {meta['brand_key'].nunique()} | files resolved: {int(meta['file_exists'].sum())}")

Rows: 1352 | sites: 189 | brands: 188 | files resolved: 1259


## 3. Image loading (raster + SVG) and foreground extraction

In [5]:
def decode_bgra(data):
    """Decode any raster format to BGRA uint8 (OpenCV first, Pillow for ICO/GIF/others)."""
    img = cv2.imdecode(np.frombuffer(data, np.uint8), cv2.IMREAD_UNCHANGED)
    if img is None and PIL_OK:
        try:
            img = cv2.cvtColor(np.array(Image.open(io.BytesIO(data)).convert("RGBA")), cv2.COLOR_RGBA2BGRA)
        except Exception:
            return None
    if img is None:
        return None
    if img.dtype == np.uint16:
        img = (img / 257).astype(np.uint8)
    if img.ndim == 2:
        img = cv2.cvtColor(img, cv2.COLOR_GRAY2BGRA)
    elif img.shape[2] == 3:
        img = cv2.cvtColor(img, cv2.COLOR_BGR2BGRA)
    return img


def looks_like_svg(path, data):
    return path.suffix.lower() == ".svg" or b"<svg" in data[:4096].lower()


def render_svg(data):
    if not SVG_RENDER_OK:
        return None, "svg_renderer_unavailable"
    for kwargs in ({"output_width": SVG_RASTER_WIDTH}, {"output_width": SVG_RASTER_WIDTH, "parent_width": 512, "parent_height": 512}):
        try:
            png = cairosvg.svg2png(bytestring=data, **kwargs)
            img = decode_bgra(png)
            if img is not None and img.shape[0] > 1 and img.shape[1] > 1:
                return img, ""
        except Exception:
            continue
    return None, "svg_render_failed"


def to_lab(bgr):
    """uint8 BGR array (N,3) -> real CIE Lab (L 0-100)."""
    arr = np.asarray(bgr, dtype=np.uint8).reshape(-1, 1, 3)
    lab = cv2.cvtColor(arr, cv2.COLOR_BGR2LAB).reshape(-1, 3).astype(np.float32)
    lab[:, 0] *= 100.0 / 255.0
    lab[:, 1:] -= 128.0
    return lab


def estimate_background(bgr):
    h, w = bgr.shape[:2]
    t = max(1, int(round(0.02 * min(h, w))))
    ring = np.concatenate([bgr[:t].reshape(-1, 3), bgr[-t:].reshape(-1, 3), bgr[:, :t].reshape(-1, 3), bgr[:, -t:].reshape(-1, 3)])
    med = np.median(ring, axis=0).astype(np.uint8)
    dist = np.linalg.norm(to_lab(ring) - to_lab(med[None])[0], axis=1)
    return med, float((dist < 6).mean())


def foreground_mask(bgra):
    bgr, alpha = bgra[..., :3], bgra[..., 3]
    h, w = alpha.shape
    corners = np.array([alpha[0, 0], alpha[0, -1], alpha[-1, 0], alpha[-1, -1]])
    info = {"corners_transparent": float((corners < 32).mean())}
    if (alpha < ALPHA_THRESHOLD).mean() > 0.005:   # a visible transparent area (corners of icons count)
        mask = ((alpha > ALPHA_THRESHOLD) * 255).astype(np.uint8)
        info.update(bg_type="transparent", bg_hex="", mask_method="alpha", bg_uniformity=np.nan, has_alpha=True)
        return mask, info
    bg, uni = estimate_background(bgr)
    info.update(bg_uniformity=uni, has_alpha=False)
    if uni >= BG_UNIFORMITY:
        lab = to_lab(bgr.reshape(-1, 3)).reshape(h, w, 3)
        dist = np.linalg.norm(lab - to_lab(bg[None])[0], axis=2)
        mask = ((dist > FG_DELTAE) * 255).astype(np.uint8)
        L, a, b = to_lab(bg[None])[0]
        chroma = math.hypot(a, b)
        bg_type = "solid_light" if (L > 92 and chroma < 8) else "solid_dark" if L < 15 else "solid_color"
        info.update(bg_type=bg_type, bg_hex="#%02X%02X%02X" % (bg[2], bg[1], bg[0]), mask_method="lab_distance")
    else:
        gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
        _, mask = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        if (mask > 0).mean() > 0.5:
            mask = 255 - mask
        info.update(bg_type="complex", bg_hex="", mask_method="otsu_gray")
    return mask, info


def preprocess(bgra):
    """Foreground mask -> tight crop -> normalised size. Returns None when no foreground is found."""
    h0, w0 = bgra.shape[:2]
    mask, info = foreground_mask(bgra)
    pts = cv2.findNonZero(mask)
    if pts is None or len(pts) < 20:
        return None
    x, y, w, h = cv2.boundingRect(pts)
    info.update(orig_w=w0, orig_h=h0, orig_aspect=w0 / h0, content_aspect_ratio=w / h,
                content_area_ratio=(w * h) / (w0 * h0), canvas_fg_ratio=float((mask > 0).mean()))
    crop_bgr, crop_mask = bgra[y:y + h, x:x + w, :3], mask[y:y + h, x:x + w]
    scale = min(math.sqrt(TARGET_AREA / (w * h)), MAX_SIDE / max(w, h))
    nw, nh = max(8, int(round(w * scale))), max(8, int(round(h * scale)))
    interp = cv2.INTER_AREA if scale < 1 else cv2.INTER_CUBIC
    bgr_n = cv2.resize(crop_bgr, (nw, nh), interpolation=interp)
    mask_n = ((cv2.resize(crop_mask, (nw, nh), interpolation=interp) > 127) * 255).astype(np.uint8)
    # remove specks
    n, lab, st, _ = cv2.connectedComponentsWithStats(mask_n, connectivity=8)
    keep = [i for i in range(1, n) if st[i, cv2.CC_STAT_AREA] >= MIN_COMPONENT_AREA_RATIO * nw * nh]
    mask_n = (np.isin(lab, keep) * 255).astype(np.uint8)
    if mask_n.sum() == 0:
        return None
    return {"bgr": bgr_n, "mask": mask_n, "info": info}

## 4. Feature extraction with OpenCV

In [6]:
def hex_of(bgr):
    b, g, r = [int(round(v)) for v in bgr]
    return "#%02X%02X%02X" % (r, g, b)


def classify_family(hex_color):
    import colorsys
    r, g, b = (int(hex_color[i:i + 2], 16) / 255 for i in (1, 3, 5))
    h, s, _ = colorsys.rgb_to_hsv(r, g, b)
    hue = h * 360
    if s < .10: return "Neutral"
    if hue < 15 or hue >= 345: return "Red"
    if hue < 45: return "Orange"
    if hue < 70: return "Yellow"
    if hue < 160: return "Green"
    if hue < 200: return "Cyan"
    if hue < 255: return "Blue"
    if hue < 290: return "Purple"
    return "Pink"


def rel_luminance(hex_color):
    ch = [int(hex_color[i:i + 2], 16) / 255 for i in (1, 3, 5)]
    lin = [c / 12.92 if c <= .04045 else ((c + .055) / 1.055) ** 2.4 for c in ch]
    return .2126 * lin[0] + .7152 * lin[1] + .0722 * lin[2]


def contrast_ratio(hex_a, hex_b):
    la, lb = rel_luminance(hex_a), rel_luminance(hex_b)
    hi, lo = max(la, lb), min(la, lb)
    return (hi + .05) / (lo + .05)


# ---- Geometry / composition ----------------------------------------------------------------------------
def composition_features(mask):
    H, W = mask.shape
    fg_area = float((mask > 0).sum())
    raw_n, raw_lab, raw_st, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)
    k = max(3, int(GROUP_GAP_RATIO * H)); k += (k % 2 == 0)
    dil = cv2.dilate(mask, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k, k)))
    ng, glab, _, _ = cv2.connectedComponentsWithStats(dil, connectivity=8)
    groups = []
    for g in range(1, ng):
        gm = (glab == g) & (mask > 0)
        area = float(gm.sum())
        if area == 0:
            continue
        x, y, w, h = cv2.boundingRect(gm.astype(np.uint8))
        ids = np.unique(raw_lab[gm]); ids = ids[ids > 0]
        boxes = raw_st[ids][:, :4]
        heights = boxes[:, 3].astype(float)
        order = np.argsort(boxes[:, 0])
        xs = boxes[order]
        gaps = [max(0, xs[i + 1, 0] - (xs[i, 0] + xs[i, 2])) for i in range(len(xs) - 1)]
        aspect = w / h
        text_like = (len(ids) >= TEXT_MIN_COMPONENTS and aspect >= TEXT_MIN_ASPECT) or aspect >= WORDMARK_ASPECT
        groups.append({"x": x, "y": y, "w": w, "h": h, "area": area, "share": area / fg_area, "n_components": len(ids),
                       "aspect": aspect, "fill": area / (w * h), "text_like": bool(text_like),
                       "height_cv": float(heights.std() / heights.mean()) if len(ids) > 1 and heights.mean() else np.nan,
                       "baseline_std": float(np.std(boxes[:, 1] + boxes[:, 3]) / h) if len(ids) > 1 else np.nan,
                       "gap_ratio": float(np.median(gaps) / h) if gaps else np.nan})
    major = [g for g in groups if g["share"] >= MIN_GROUP_SHARE]
    texts = sorted([g for g in major if g["text_like"]], key=lambda g: -g["area"])
    symbols = sorted([g for g in major if not g["text_like"]], key=lambda g: -g["area"])
    if texts and symbols:
        comp = "Combination (symbol + wordmark)"
    elif texts:
        comp = "Wordmark only"
    elif symbols:
        comp = "Symbol / emblem only"
    else:
        comp = "Undetermined"
    f = {"composition_type": comp, "n_raw_components": raw_n - 1, "n_groups": len(major), "n_minor_groups": len(groups) - len(major),
         "n_text_groups": len(texts), "n_symbol_groups": len(symbols), "symbol_layout": np.nan, "symbol_area_share": np.nan,
         "symbol_to_text_height_ratio": np.nan, "lockup_gap_ratio": np.nan, "text_letter_components": np.nan,
         "text_baseline_std": np.nan, "text_gap_ratio": np.nan, "text_height_cv": np.nan}
    if texts:
        t = texts[0]
        f.update(text_letter_components=t["n_components"], text_baseline_std=t["baseline_std"], text_gap_ratio=t["gap_ratio"], text_height_cv=t["height_cv"])
    if texts and symbols:
        s, t = symbols[0], texts[0]
        sx, sy, tx, ty = s["x"] + s["w"] / 2, s["y"] + s["h"] / 2, t["x"] + t["w"] / 2, t["y"] + t["h"] / 2
        dx, dy = (sx - tx) / W, (sy - ty) / H
        if abs(dx) >= abs(dy):
            f["symbol_layout"] = "Symbol left of text" if dx < 0 else "Symbol right of text"
        else:
            f["symbol_layout"] = "Symbol above text" if dy < 0 else "Symbol below text"
        gap_x = max(0, max(s["x"], t["x"]) - min(s["x"] + s["w"], t["x"] + t["w"]))
        gap_y = max(0, max(s["y"], t["y"]) - min(s["y"] + s["h"], t["y"] + t["h"]))
        f.update(symbol_area_share=s["area"] / (s["area"] + t["area"]), symbol_to_text_height_ratio=s["h"] / t["h"],
                 lockup_gap_ratio=(gap_x if gap_x > 0 else gap_y) / H)
    return f, groups


# ---- Shape / stroke / symmetry ---------------------------------------------------------------------------
def shape_features(mask):
    H, W = mask.shape
    m = mask > 0
    fg_area = float(m.sum())
    cnts, hier = cv2.findContours(mask, cv2.RETR_CCOMP, cv2.CHAIN_APPROX_NONE)
    hier = hier[0] if hier is not None else []
    outer = [i for i in range(len(cnts)) if hier[i][3] == -1]
    holes = [i for i in range(len(cnts)) if hier[i][3] != -1]
    areas = {i: cv2.contourArea(cnts[i]) for i in outer}
    main = max(outer, key=lambda i: areas[i]) if outer else None
    f = {"n_outer_shapes": len(outer), "n_holes": len(holes)}
    perim_total = sum(cv2.arcLength(c, True) for c in cnts)
    f["perimeter_norm"] = perim_total / math.sqrt(fg_area)
    pts = cv2.findNonZero(mask)
    hull_all = cv2.contourArea(cv2.convexHull(pts)) if pts is not None else np.nan
    f["global_solidity"] = fg_area / hull_all if hull_all else np.nan
    if main is not None:
        c = cnts[main]; a = max(areas[main], 1.0); p = cv2.arcLength(c, True)
        hull = cv2.contourArea(cv2.convexHull(c))
        f.update(circularity_main=min(1.0, 4 * math.pi * a / (p * p)) if p else np.nan, solidity_main=a / hull if hull else np.nan,
                 vertices_per_100px=len(cv2.approxPolyDP(c, 0.01 * p, True)) / (p / 100) if p else np.nan)
        # container: nested content inside the largest outer contour
        bx, by, bw, bh = cv2.boundingRect(c)
        spans_logo = (bw * bh) / (W * H) >= CONTAINER_BBOX_RATIO
        inner = 0.0; n_nested = 0
        for i in outer:
            if i == main:
                continue
            mo = cv2.moments(cnts[i])
            if mo["m00"] > 0 and cv2.pointPolygonTest(c, (mo["m10"] / mo["m00"], mo["m01"] / mo["m00"]), False) >= 0:
                inner += areas[i]; n_nested += 1
        f.update(n_nested_shapes=n_nested, nested_share=inner / fg_area, container_flag=bool(spans_logo and inner / fg_area >= CONTAINER_INNER_SHARE))
    corners = cv2.goodFeaturesToTrack(mask, maxCorners=800, qualityLevel=0.1, minDistance=max(3, H // 40))
    f["corner_density"] = (0 if corners is None else len(corners)) / (perim_total / 100) if perim_total else np.nan
    # symmetry and balance
    f["sym_h"] = (m & m[:, ::-1]).sum() / max((m | m[:, ::-1]).sum(), 1)
    f["sym_v"] = (m & m[::-1, :]).sum() / max((m | m[::-1, :]).sum(), 1)
    f["sym_point"] = (m & m[::-1, ::-1]).sum() / max((m | m[::-1, ::-1]).sum(), 1)
    ys, xs = np.nonzero(m)
    f["centroid_x_offset"], f["centroid_y_offset"] = xs.mean() / W - .5, ys.mean() / H - .5
    for name, sl in (("tl", (slice(0, H // 2), slice(0, W // 2))), ("tr", (slice(0, H // 2), slice(W // 2, W))),
                     ("bl", (slice(H // 2, H), slice(0, W // 2))), ("br", (slice(H // 2, H), slice(W // 2, W)))):
        f[f"weight_{name}"] = m[sl].sum() / fg_area
    # stroke weight from the distance transform ridge
    dt = cv2.distanceTransform(mask, cv2.DIST_L2, 3)
    ridge = (dt == cv2.dilate(dt, np.ones((3, 3), np.uint8))) & (dt > 0.5)
    r = dt[ridge]
    f["stroke_weight_ratio"] = 2 * np.median(r) / H if len(r) else np.nan
    f["stroke_uniformity_cv"] = float(r.std() / r.mean()) if len(r) and r.mean() else np.nan
    f["max_thickness_ratio"] = 2 * dt.max() / H
    f["fg_fill_ratio"] = fg_area / (W * H)
    return f


# ---- Color ---------------------------------------------------------------------------------------------------------
def seg_dist(p, a, b):
    ab = b - a
    t = np.clip(np.dot(p - a, ab) / max(np.dot(ab, ab), 1e-9), 0, 1)
    return float(np.linalg.norm(p - (a + t * ab)))


def merge_tints(pal, bg_lab):
    """Merge minor palette colors that lie on the line between a larger color and the background (anti-aliasing tints)."""
    out = []
    for bgr_m, share, lab_m in pal:
        merged = False
        if out and share < TINT_MAX_SHARE:
            for item in out:
                if seg_dist(lab_m, item[2], bg_lab) < TINT_DELTAE:
                    item[1] += share; merged = True; break
        if not merged:
            out.append([bgr_m, share, lab_m])
    return [tuple(i) for i in out]


def palette_features(bgr, mask, bg_hex=""):
    H = mask.shape[0]
    core = cv2.erode(mask, np.ones((1 + 2 * max(1, H // 128),) * 2, np.uint8))
    sel = core > 0 if core.sum() else mask > 0
    px = bgr[sel]
    if len(px) > MAX_COLOR_PIXELS:
        px = px[RNG.choice(len(px), MAX_COLOR_PIXELS, replace=False)]
    lab = to_lab(px)
    uniq = len(np.unique(px, axis=0))
    k = max(1, min(COLOR_K, uniq))
    if k == 1:
        labels, centers = np.zeros(len(px), int), lab.mean(0, keepdims=True)
    else:
        _, labels, centers = cv2.kmeans(lab, k, None, (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, .5), 3, cv2.KMEANS_PP_CENTERS)
        labels = labels.ravel()
    parent = list(range(k))
    def find(i):
        while parent[i] != i:
            parent[i] = parent[parent[i]]; i = parent[i]
        return i
    for i, j in combinations(range(k), 2):
        if np.linalg.norm(centers[i] - centers[j]) < MERGE_DELTAE:
            parent[find(i)] = find(j)
    groups = {}
    for i in range(k):
        groups.setdefault(find(i), []).append(i)
    pal, resid = [], 0.0
    for members in groups.values():
        idx = np.isin(labels, members)
        if not idx.any():
            continue
        gl = lab[idx]
        resid += np.linalg.norm(gl - gl.mean(0), axis=1).sum()
        pal.append((px[idx].mean(0), idx.mean(), gl.mean(0)))
    spread = resid / len(px)
    pal = sorted(pal, key=lambda p: -p[1])
    if bg_hex:
        bg_lab = to_lab(np.array([[int(bg_hex[5:7], 16), int(bg_hex[3:5], 16), int(bg_hex[1:3], 16)]]))[0]
        pal = merge_tints(pal, bg_lab)
    pal = [p for p in pal if p[1] >= MIN_COLOR_SHARE] or pal[:1]
    tot = sum(p[1] for p in pal)
    pal = [(hex_of(p[0]), p[1] / tot, p[2]) for p in pal]
    chroma = [math.hypot(p[2][1], p[2][2]) for p in pal]
    dom_hex = pal[0][0]
    accent = pal[int(np.argmax(chroma))][0]
    n = len(pal)
    f = {"n_colors": n, "color_mode": "Monochrome (1 color)" if n == 1 else "Two colors" if n == 2 else "Three colors" if n == 3 else "Four or more colors",
         "dominant_hex": dom_hex, "dominant_family": classify_family(dom_hex), "dominant_share": pal[0][1],
         "accent_hex": accent, "accent_family": classify_family(accent), "max_chroma": max(chroma),
         "is_achromatic": bool(max(chroma) < ACHROMATIC_CHROMA), "core_color_spread": spread,
         "unique_colors_4bit": len(np.unique((px >> 4).astype(np.int32) @ np.array([256, 16, 1]))),
         "fg_lightness": float(np.average([p[2][0] for p in pal], weights=[p[1] for p in pal])),
         "contrast_on_white": contrast_ratio(dom_hex, "#FFFFFF"), "contrast_on_black": contrast_ratio(dom_hex, "#000000"),
         "palette": ";".join(f"{h}:{s:.3f}" for h, s, _ in pal)}
    f["gradient_flag"] = bool(spread > GRADIENT_SPREAD)
    f["photographic_flag"] = bool(f["unique_colors_4bit"] > PHOTO_UNIQUE_COLORS)
    hsv = cv2.cvtColor(px.reshape(-1, 1, 3), cv2.COLOR_BGR2HSV).reshape(-1, 3)
    f["mean_saturation"] = float(hsv[:, 1].mean() / 255)
    return f


# ---- Scalability ----------------------------------------------------------------------------------------------------
def scalability_features(mask):
    H, W = mask.shape
    ref = mask > 0
    n_ref = cv2.connectedComponents(mask)[0] - 1
    mf = mask.astype(np.float32) / 255
    f = {}
    for T in SMALL_HEIGHTS:
        if H <= T:
            f[f"retention_iou_{T}"] = np.nan; f[f"component_retention_{T}"] = np.nan
            continue
        small = cv2.resize(mf, (max(1, round(W * T / H)), T), interpolation=cv2.INTER_AREA)
        back = cv2.resize(small, (W, H), interpolation=cv2.INTER_LINEAR) > .5
        f[f"retention_iou_{T}"] = (back & ref).sum() / max((back | ref).sum(), 1)
        n_after = cv2.connectedComponents((small > .5).astype(np.uint8))[0] - 1
        f[f"component_retention_{T}"] = min(n_after, n_ref) / n_ref if n_ref else np.nan
    return f


# ---- SVG structure ---------------------------------------------------------------------------------------------------
def svg_structure(text):
    low = text.lower()
    vb = re.search(r'viewbox\s*=\s*["\']\s*([-\d.eE]+)[ ,]+([-\d.eE]+)[ ,]+([-\d.eE]+)[ ,]+([-\d.eE]+)', text, re.I)
    vb_aspect = np.nan
    if vb:
        try:
            vb_aspect = float(vb.group(3)) / float(vb.group(4))
        except (ValueError, ZeroDivisionError):
            pass
    fills = re.findall(r'(?:fill|stroke)\s*[=:]\s*["\']?\s*(#[0-9a-fA-F]{3,8}|rgb\([^)]*\))', text)
    d_len = sum(len(m) for m in re.findall(r'\sd\s*=\s*"([^"]*)"', text))
    return {"svg_bytes": len(text.encode("utf-8", "ignore")), "svg_paths": low.count("<path"),
            "svg_basic_shapes": sum(low.count(t) for t in ("<circle", "<ellipse", "<rect", "<polygon", "<polyline", "<line")),
            "svg_text_elements": low.count("<text") + low.count("<tspan"), "svg_groups": low.count("<g"),
            "svg_gradients": low.count("lineargradient") + low.count("radialgradient"), "svg_clip_or_mask": low.count("<clippath") + low.count("<mask"),
            "svg_embedded_raster": int("<image" in low or "data:image" in low), "svg_uses_css_classes": int("class=" in low or "<style" in low),
            "svg_uses_currentcolor": int("currentcolor" in low), "svg_distinct_fills": len({f.lower() for f in fills}),
            "svg_path_data_chars": d_len, "svg_viewbox_aspect": vb_aspect, "svg_has_viewbox": int(vb is not None)}

## 5. Run the extraction over every downloaded asset

In [7]:
PROC, FEATURES = {}, []


def analyze_asset(row):
    rec = {"asset_id": row.asset_id, "analysis_status": "ok", "render_note": ""}
    path = row.file_path
    try:
        data = Path(path).read_bytes()
    except Exception as e:
        rec.update(analysis_status="read_failed", render_note=str(e)[:100]); return rec
    is_svg = looks_like_svg(Path(path), data)
    rec["is_svg"] = is_svg
    if is_svg:
        rec.update(svg_structure(data.decode("utf-8", "ignore")))
        bgra, note = render_svg(data)
    else:
        bgra, note = decode_bgra(data), ""
        note = "" if bgra is not None else "decode_failed"
    if bgra is None:
        rec.update(analysis_status="load_failed", render_note=note); return rec
    pre = preprocess(bgra)
    if pre is None:
        rec.update(analysis_status="empty_mask"); return rec
    mask, bgr = pre["mask"], pre["bgr"]
    comp, groups = composition_features(mask)
    rec.update(pre["info"])
    rec.update(comp); rec.update(shape_features(mask)); rec.update(palette_features(bgr, mask, pre["info"].get("bg_hex", ""))); rec.update(scalability_features(mask))
    rec["norm_w"], rec["norm_h"] = mask.shape[1], mask.shape[0]
    PROC[row.asset_id] = {"bgr": bgr, "mask": mask, "groups": groups}
    rgba = np.dstack([bgr, mask])
    cv2.imencode(".png", rgba)[1].tofile(str(PROCESSED / f"{row.asset_id:05d}_{row.brand_key}_{row.role}.png"))
    return rec


todo = meta[meta["is_success"] & meta["file_exists"]]
print(f"Analysing {len(todo)} assets ...")
for i, row in enumerate(todo.itertuples(index=False), 1):
    FEATURES.append(analyze_asset(row))
    if i % 50 == 0:
        print(f"  {i}/{len(todo)}")

feat = pd.DataFrame(FEATURES)
if feat.empty:
    raise RuntimeError("No assets could be analysed. Check the inventory validation output.")
assets = meta.merge(feat, on="asset_id", how="left")
assets["analysis_status"] = assets["analysis_status"].fillna("not_downloaded")
for status, n in assets["analysis_status"].value_counts().items():
    if status not in ("ok", "not_downloaded"):
        validate("Inventory", f"Assets with analysis status '{status}'", "PARTIAL", "Excluded from OpenCV features.", int(n))
print(assets["analysis_status"].value_counts().to_string())

Analysing 1259 assets ...
  50/1259
  100/1259
  150/1259
  200/1259
  250/1259
  300/1259
  350/1259
  400/1259
  450/1259
  500/1259
  550/1259
  600/1259
  650/1259
  700/1259
  750/1259
  800/1259
  850/1259
  900/1259
  950/1259
  1000/1259
  1050/1259
  1100/1259
  1150/1259
  1200/1259
  1250/1259
analysis_status
ok                734
load_failed       524
not_downloaded     93
empty_mask          1


## 6. Derived classifications, quality flags and analysis sets

In [8]:
a = assets.copy()
ok = a["analysis_status"].eq("ok")


def flags(r):
    out = []
    if r["analysis_status"] != "ok": return r["analysis_status"]
    if r.get("photographic_flag"): out.append("photographic_or_complex")
    if r.get("bg_type") == "complex": out.append("complex_background")
    if r.get("orig_h", 99) < 24 and r["role"] in ("primary", "variant"): out.append("tiny_original")
    if r.get("orig_w", 0) >= 1500 or r.get("orig_h", 0) >= 1500: out.append("very_large_original")
    if r.get("content_aspect_ratio", 1) > 12 or r.get("content_aspect_ratio", 1) < 0.15: out.append("extreme_aspect")
    if r.get("is_svg") and (r.get("svg_uses_css_classes") or r.get("svg_uses_currentcolor")): out.append("svg_css_dependent_render")
    if r.get("is_svg") and r.get("svg_embedded_raster"): out.append("svg_embeds_raster")
    return ";".join(out)


a["quality_flags"] = a.apply(flags, axis=1)
a["include_in_analysis"] = ok & ~(a["photographic_flag"].fillna(False).astype(bool) & EXCLUDE_PHOTOGRAPHIC)
a["aspect_class"] = pd.cut(a["content_aspect_ratio"], bins=ASPECT_BINS, labels=ASPECT_LABELS, right=False)


def terciles(series, names):
    s = pd.to_numeric(series, errors="coerce")
    if s.notna().sum() < 6 or s.nunique() < 3:
        return pd.Series(np.nan, index=series.index, dtype=object)
    return pd.qcut(s, 3, labels=names, duplicates="drop").astype(object)


logos_ok = a["include_in_analysis"] & a["role"].isin(["primary", "variant"])
for col, name, labels in (("stroke_weight_ratio", "stroke_weight_class", ["Light stroke", "Medium stroke", "Heavy stroke"]),
                          ("corner_density", "angularity_class", ["Rounded / smooth", "Mixed", "Angular"]),
                          ("perimeter_norm", "complexity_class", ["Simple", "Moderate", "Intricate"])):
    a[name] = pd.Series([None] * len(a), index=a.index, dtype=object)
    a.loc[logos_ok, name] = terciles(a.loc[logos_ok, col], labels)

# Representative logo per site: primary if usable, otherwise best successful variant (optional)
cand = a[logos_ok].copy()
cand["pref"] = np.where(cand["role"] == "primary", 0, 1)
if not FALLBACK_TO_VARIANT:
    cand = cand[cand["role"] == "primary"]
rep = cand.sort_values(["domain_key", "pref", "logo_rank"]).groupby("domain_key").head(1).copy()
rep["representative_is_primary"] = rep["role"] == "primary"
sites_without_rep = n_sites - rep["domain_key"].nunique()
if sites_without_rep:
    validate("Inventory", "Sites without a usable logo", "PARTIAL", f"{sites_without_rep} sites have no analysable logo.", sites_without_rep)
if (~rep["representative_is_primary"]).any():
    validate("Inventory", "Sites using a variant as representative logo", "INFO", f"{int((~rep['representative_is_primary']).sum())} sites.", int((~rep["representative_is_primary"]).sum()))

rep["brand_pref"] = rep["confidence_score"].fillna(0)
rep_brand = rep.sort_values(["brand_key", "pref", "brand_pref"], ascending=[True, True, False]).groupby("brand_key").head(1)
BASIS = rep_brand if DEDUP_BY_BRAND else rep
BASIS_LABEL = "one logo per brand" if DEDUP_BY_BRAND else "one logo per site"
print(f"Representative logos: {len(rep)} sites | {len(rep_brand)} brands | analysis basis = {BASIS_LABEL} (n = {len(BASIS)})")
if len(BASIS) < 30:
    validate("Archetypes", "Small analysis basis", "INFO", f"Only {len(BASIS)} logos in the basis; percentages are unstable.", len(BASIS))

# optional market segment
if SEGMENTS_CSV.exists():
    seg = pd.read_csv(SEGMENTS_CSV, encoding="utf-8-sig")
    if {"domain", "market_segment"} <= set(seg.columns):
        seg["domain_key"] = seg["domain"].astype(str).str.lower().str.replace(r"^www\.", "", regex=True)
        seg = seg[seg["market_segment"].astype(str).str.strip().replace("nan", "") != ""]
        smap = seg.drop_duplicates("domain_key").set_index("domain_key")["market_segment"]
        BASIS = BASIS.assign(market_segment=BASIS["domain_key"].map(smap))
        rep = rep.assign(market_segment=rep["domain_key"].map(smap))
        if BASIS["market_segment"].notna().sum() == 0:
            validate("Composition", "Market segments", "EMPTY", "market_classification_template.csv has no filled segments.")
    else:
        validate("Composition", "Market segments", "MISSING", "Segments file lacks domain / market_segment columns.")
else:
    validate("Composition", "Market segments", "MISSING", f"{SEGMENTS_CSV} not found; stratified tables skipped.")

# ---- Site funnel: what happened to every site --------------------------------------------------------------
def site_outcome(g):
    logos = g[g["asset_type"] == "logo"]
    if logos.empty:
        return "No logo candidate detected"
    succ = logos[logos["is_success"]]
    if succ.empty:
        return "All logo downloads failed (" + ", ".join(sorted(set(logos["download_status"].astype(str)))[:3]) + ")"
    present = succ[succ["file_exists"]]
    if present.empty:
        return "Downloaded but file not found on disk (check PROJECT_ROOT)"
    okk = present[present["analysis_status"] == "ok"]
    if okk.empty:
        return "Not analysable (" + ", ".join(sorted(set(present["analysis_status"].astype(str)))) + ")"
    if not okk["include_in_analysis"].any():
        return "Excluded by photographic / complex filter"
    return "Included"


rows_ = []
for dk, g in a.groupby("domain_key"):
    rows_.append({"Domain": dk, "Brand": g["brand_key"].iloc[0], "Outcome": site_outcome(g), "Logo rows (n)": int((g["asset_type"] == "logo").sum()),
                  "Primary logo status": g.loc[g["role"] == "primary", "download_status"].astype(str).iloc[0] if (g["role"] == "primary").any() else "none",
                  "Used variant instead of primary": bool(dk in set(rep.loc[~rep["representative_is_primary"], "domain_key"]))})
site_status = pd.DataFrame(rows_)
fun = site_status["Outcome"].str.replace(r" \(.*", "", regex=True).value_counts().rename_axis("Outcome").reset_index(name="Sites (n)")
fun["Sites (%)"] = [pct(v, n_sites) for v in fun["Sites (n)"]]; fun["Base sites (n)"] = n_sites
save_table(fun, "inventory_site_funnel.csv", "Inventory", "How many sites end up in the analysis and why the others are lost.")
save_table(site_status.sort_values("Outcome"), "inventory_site_status.csv", "Inventory", "Outcome per site (included or reason for exclusion).")
print("\nSITE FUNNEL (sites in metadata =", n_sites, ")")
print(fun.to_string(index=False))
lost = int((site_status["Outcome"] != "Included").sum())
if lost:
    validate("Inventory", "Sites not included in the analysis", "PARTIAL", f"{lost} of {n_sites} sites; see tables/inventory_site_status.csv for the reason of each.", lost)

a.to_csv(OUT / "logo_features_all_assets.csv", index=False, encoding="utf-8-sig")
rep.drop(columns=["file_path"]).to_csv(OUT / "logo_features_representative.csv", index=False, encoding="utf-8-sig")
MANIFEST += [{"Dimension": "Inventory", "Type": "table", "Filename": "logo_features_all_assets.csv", "Rows": len(a), "Description": "Every asset with all extracted features and flags."},
             {"Dimension": "Inventory", "Type": "table", "Filename": "logo_features_representative.csv", "Rows": len(rep), "Description": "One representative logo per site with all features."}]

Representative logos: 71 sites | 71 brands | analysis basis = one logo per site (n = 71)

SITE FUNNEL (sites in metadata = 189 )
                   Outcome  Sites (n)  Sites (%)  Base sites (n)
            Not analysable         85      44.97             189
                  Included         71      37.57             189
No logo candidate detected         21      11.11             189
 All logo downloads failed         12       6.35             189


## 7. Tables

In [9]:
# ---- Inventory -----------------------------------------------------------------------------------------
N_ROWS = len(meta)
inv = pd.DataFrame([
    ("Asset rows in metadata", N_ROWS, N_ROWS), ("Distinct sites (domains)", n_sites, np.nan), ("Distinct brands", meta["brand_key"].nunique(), np.nan),
    ("Rows with successful download", int(meta["is_success"].sum()), N_ROWS), ("Assets analysed with OpenCV", int(ok.sum()), N_ROWS),
    ("Sites with a usable representative logo", rep["domain_key"].nunique(), n_sites),
    ("Brands in the analysis basis", len(rep_brand), meta["brand_key"].nunique())], columns=["Metric", "Count (n)", "Base (n)"])
inv["Share (%)"] = [pct(c, b) if pd.notna(b) else np.nan for c, b in zip(inv["Count (n)"], inv["Base (n)"])]
save_table(inv, "inventory_overview.csv", "Inventory", "Headline counts of the logo collection.")
st = dist_table(meta["download_status"].fillna("missing"), "Download status"); save_table(st, "inventory_download_status.csv", "Inventory", "Download status of every asset row.")
save_table(pd.crosstab(meta["asset_type"], meta["is_success"].map({True: "Downloaded", False: "Failed"})).reset_index().rename(columns={"asset_type": "Asset type"}),
           "inventory_asset_types.csv", "Inventory", "Asset types by download outcome.")
fmt = meta[meta["is_success"]]["format_detected"].fillna("unknown").value_counts().rename_axis("Detected format").reset_index(name="Assets (n)")
fmt["Assets (%)"] = [pct(v, fmt["Assets (n)"].sum()) for v in fmt["Assets (n)"]]; fmt["Base (n)"] = fmt["Assets (n)"].sum()
save_table(fmt, "inventory_formats.csv", "Inventory", "File formats of downloaded assets.")
per_site = meta.groupby("domain_key").agg(logo_rows=("asset_type", lambda s: int((s == "logo").sum())), has_favicon=("asset_type", lambda s: int((s == "favicon").sum()) > 0),
                                          has_apple_icon=("asset_type", lambda s: int((s == "apple_touch_icon").sum()) > 0)).reset_index()
save_table(pd.DataFrame([describe_cols(per_site, ["logo_rows"], {"logo_rows": "Logo candidates per site"}).iloc[0].to_dict()]), "inventory_logos_per_site.csv", "Inventory", "Logo candidates detected per site.")
dups = meta[meta["sha256"].notna() & (meta["sha256"].astype(str) != "")].groupby("sha256").agg(assets=("asset_id", "size"), sites=("domain_key", "nunique"),
                                                                                               brands=("brand_key", "nunique"), example_domains=("domain_key", lambda s: "; ".join(sorted(set(s))[:5]))).reset_index()
dups = dups[dups["assets"] > 1].rename(columns={"sha256": "SHA-256", "assets": "Assets (n)", "sites": "Sites (n)", "brands": "Brands (n)", "example_domains": "Example domains"}).sort_values("Assets (n)", ascending=False)
save_table(dups, "inventory_duplicate_files.csv", "Inventory", "Byte-identical assets shared by several sites (e.g. regional domains of one brand).")
fl = a.loc[a["quality_flags"].fillna("") != "", "quality_flags"].str.split(";").explode().value_counts().rename_axis("Quality flag").reset_index(name="Assets (n)")
fl["Assets (%)"] = [pct(v, int(ok.sum())) for v in fl["Assets (n)"]]; fl["Base analysed assets (n)"] = int(ok.sum())
save_table(fl, "inventory_quality_flags.csv", "Inventory", "Quality flags raised during analysis (flagged assets stay in the dataset unless photographic).")
role_t = a.groupby("role").agg(rows=("asset_id", "size"), analysed=("include_in_analysis", "sum")).reset_index().rename(columns={"role": "Asset role", "rows": "Rows (n)", "analysed": "Analysed (n)"})
save_table(role_t, "inventory_roles.csv", "Inventory", "Asset roles (primary logo, variants, favicon, apple touch icon).")

# ---- Geometry ----------------------------------------------------------------------------------------------
G = BASIS
geo_labels = {"orig_w": "Original width (px)", "orig_h": "Original height (px)", "content_aspect_ratio": "Content aspect ratio (W/H, tight crop)",
              "content_area_ratio": "Content area / canvas area", "fg_fill_ratio": "Foreground fill of bounding box"}
save_table(describe_cols(G, list(geo_labels), geo_labels), "logo_geometry_summary.csv", "Geometry", f"Geometry statistics ({BASIS_LABEL}).")
ori = dist_table(G["aspect_class"].astype(object), "Aspect class"); ori["_o"] = ori["Aspect class"].map({l: i for i, l in enumerate(ASPECT_LABELS)}); R["orientation"] = ori.sort_values("_o").drop(columns="_o")
save_table(R["orientation"], "logo_orientation.csv", "Geometry", "Logo orientation classes from the tight-crop aspect ratio.")
save_table(dist_table(G["bg_type"], "Original background"), "logo_background_types.csv", "Geometry", "Original background type of the logo file.")
fmt_b = dist_table(G["format_detected"].fillna("unknown"), "File format"); save_table(fmt_b, "logo_file_formats.csv", "Geometry", "File format of the representative logos.")

# ---- Composition ----------------------------------------------------------------------------------------------
R["comp"] = dist_table(G["composition_type"], "Composition type")
save_table(R["comp"], "logo_composition_types.csv", "Composition", f"Heuristic composition type ({BASIS_LABEL}). Validate with the overlay sheet.")
combo = G[G["composition_type"] == "Combination (symbol + wordmark)"]
R["layout"] = dist_table(combo["symbol_layout"], "Symbol position (combination logos)")
save_table(R["layout"], "logo_symbol_layout.csv", "Composition", "Symbol position relative to the wordmark in combination logos.")
comp_labels = {"n_groups": "Major visual groups per logo", "n_text_groups": "Text-like groups", "n_symbol_groups": "Symbol-like groups", "n_raw_components": "Connected components",
               "symbol_area_share": "Symbol share of (symbol + text) area", "symbol_to_text_height_ratio": "Symbol height / wordmark height",
               "lockup_gap_ratio": "Gap between symbol and wordmark / logo height", "text_letter_components": "Letter components in wordmark",
               "text_baseline_std": "Wordmark baseline deviation / height", "text_gap_ratio": "Median letter gap / wordmark height", "text_height_cv": "Letter height variation (CV)"}
save_table(describe_cols(G, list(comp_labels), comp_labels), "logo_composition_metrics.csv", "Composition", "Quantitative composition metrics (combination metrics only where applicable).")
cont = pd.DataFrame([{"Metric": "Closed outer shape containing nested content (container / badge)", "Logos (n)": int(G["container_flag"].fillna(False).sum()),
                      "Logos (%)": pct(G["container_flag"].fillna(False).sum(), G["container_flag"].notna().sum()), "Base (n)": int(G["container_flag"].notna().sum())},
                     {"Metric": "Logos with internal holes / counters", "Logos (n)": int((G["n_holes"] > 0).sum()), "Logos (%)": pct((G["n_holes"] > 0).sum(), G["n_holes"].notna().sum()), "Base (n)": int(G["n_holes"].notna().sum())}])
save_table(cont, "logo_container_and_holes.csv", "Composition", "Container shapes and counters.")
save_table(crosstab_pct(G, "composition_type", "aspect_class", "Composition type", "Aspect class"), "logo_composition_by_orientation.csv", "Composition", "Composition type vs orientation.")
save_table(crosstab_pct(G, "composition_type", "color_mode", "Composition type", "Color mode"), "logo_composition_by_color_mode.csv", "Composition", "Composition type vs color mode.")
if "market_segment" in G.columns and G["market_segment"].notna().any():
    save_table(crosstab_pct(G, "market_segment", "composition_type", "Market segment", "Composition type"), "logo_composition_by_segment.csv", "Composition", "Composition type by market segment.")

# manual validation of the heuristic
tmpl = rep[["domain_key", "brand_key", "composition_type"]].rename(columns={"composition_type": "composition_predicted"}).assign(composition_manual="")
tmpl.to_csv(OUT / "manual_labels_template.csv", index=False, encoding="utf-8-sig")
MANIFEST.append({"Dimension": "Composition", "Type": "table", "Filename": "manual_labels_template.csv", "Rows": len(tmpl), "Description": "Template to label compositions manually; save as manual_labels.csv and re-run."})
if MANUAL_LABELS_CSV.exists():
    ml = pd.read_csv(MANUAL_LABELS_CSV, encoding="utf-8-sig")
    if {"domain_key", "composition_manual"} <= set(ml.columns):
        ml = ml[ml["composition_manual"].fillna("").astype(str).str.strip() != ""].merge(rep[["domain_key", "composition_type"]], on="domain_key")
        if len(ml):
            ml["Agrees"] = ml["composition_manual"].str.strip().str.lower() == ml["composition_type"].str.lower()
            save_table(pd.DataFrame([{"Labelled logos (n)": len(ml), "Heuristic agrees (n)": int(ml["Agrees"].sum()), "Heuristic agrees (%)": pct(ml["Agrees"].sum(), len(ml))}]),
                       "logo_composition_heuristic_agreement.csv", "Composition", "Agreement between the heuristic and manual labels.")
            save_table(pd.crosstab(ml["composition_manual"], ml["composition_type"]).reset_index().rename(columns={"composition_manual": "Manual label"}),
                       "logo_composition_confusion.csv", "Composition", "Manual label (rows) vs heuristic (columns).")
else:
    validate("Composition", "Manual composition labels", "MISSING", f"{MANUAL_LABELS_CSV} not found; heuristic accuracy cannot be reported. Use manual_labels_template.csv.")

# ---- Color ------------------------------------------------------------------------------------------------------------
R["cmode"] = dist_table(G["color_mode"], "Color mode"); save_table(R["cmode"], "logo_color_modes.csv", "Color", "Number of effective colors in the logo (after merging similar tones).")
R["cfam"] = dist_table(G["dominant_family"], "Dominant color family"); save_table(R["cfam"], "logo_dominant_families.csv", "Color", "Color family of the dominant logo color.")
R["afam"] = dist_table(G.loc[~G["is_achromatic"].fillna(True).astype(bool), "accent_family"], "Accent color family (chromatic logos)")
save_table(R["afam"], "logo_accent_families.csv", "Color", "Family of the most saturated palette color, chromatic logos only.")
col_labels = {"n_colors": "Effective colors", "mean_saturation": "Mean saturation (0-1)", "fg_lightness": "Foreground lightness L* (0-100)", "max_chroma": "Maximum palette chroma (Lab)",
              "core_color_spread": "Color spread inside palette colors (deltaE)", "contrast_on_white": "Dominant color contrast on white (WCAG)", "contrast_on_black": "Dominant color contrast on black (WCAG)",
              "dominant_share": "Dominant color share of foreground"}
save_table(describe_cols(G, list(col_labels), col_labels), "logo_color_summary.csv", "Color", "Color statistics.")
flags_c = pd.DataFrame([{"Metric": m, "Logos (n)": int(G[c].fillna(False).astype(bool).sum()), "Logos (%)": pct(G[c].fillna(False).astype(bool).sum(), G[c].notna().sum()), "Base (n)": int(G[c].notna().sum())}
                        for m, c in (("Achromatic logos (black / white / grey)", "is_achromatic"), ("Gradient detected", "gradient_flag"))])
flags_c = pd.concat([flags_c, pd.DataFrame([{"Metric": "Dominant color contrast on white >= 4.5 (WCAG AA text)", "Logos (n)": int((G["contrast_on_white"] >= 4.5).sum()), "Logos (%)": pct((G["contrast_on_white"] >= 4.5).sum(), G["contrast_on_white"].notna().sum()), "Base (n)": int(G["contrast_on_white"].notna().sum())},
                                           {"Metric": "Dominant color contrast on black >= 4.5 (WCAG AA text)", "Logos (n)": int((G["contrast_on_black"] >= 4.5).sum()), "Logos (%)": pct((G["contrast_on_black"] >= 4.5).sum(), G["contrast_on_black"].notna().sum()), "Base (n)": int(G["contrast_on_black"].notna().sum())}])])
save_table(flags_c, "logo_color_flags.csv", "Color", "Achromatic, gradient and contrast indicators.")
R["lightness"] = dist_table(pd.cut(G["fg_lightness"], [-1, 35, 65, 101], labels=["Dark foreground (L* < 35)", "Mid-tone foreground", "Light foreground (L* >= 65)"]).astype(object), "Foreground lightness class")
save_table(R["lightness"], "logo_foreground_lightness.csv", "Color", "Foreground lightness classes.")
pal = G[["brand_key", "domain_key", "palette"]].copy()
for i in range(3):
    pal[f"Color {i + 1} (HEX)"] = pal["palette"].apply(lambda s, i=i: s.split(";")[i].split(":")[0] if isinstance(s, str) and len(s.split(";")) > i else "")
    pal[f"Color {i + 1} share"] = pal["palette"].apply(lambda s, i=i: float(s.split(";")[i].split(":")[1]) if isinstance(s, str) and len(s.split(";")) > i else np.nan)
save_table(pal.drop(columns="palette").rename(columns={"brand_key": "Brand", "domain_key": "Domain"}), "logo_palettes_by_brand.csv", "Color", "Top three palette colors and shares per logo.")
save_table(crosstab_pct(G, "dominant_family", "composition_type", "Dominant color family", "Composition type"), "logo_color_family_by_composition.csv", "Color", "Dominant color family vs composition type.")

# ---- Shape and stroke -----------------------------------------------------------------------------------------------------
shp = {"circularity_main": "Circularity of largest shape (0-1)", "solidity_main": "Solidity of largest shape (0-1)", "global_solidity": "Global solidity (foreground / convex hull)",
       "corner_density": "Corners per 100 px of contour", "vertices_per_100px": "Polygon vertices per 100 px of contour", "perimeter_norm": "Contour length / sqrt(area)",
       "stroke_weight_ratio": "Stroke weight / logo height", "stroke_uniformity_cv": "Stroke thickness variation (CV)", "max_thickness_ratio": "Maximum thickness / logo height",
       "n_outer_shapes": "Outer shapes", "n_holes": "Holes / counters"}
save_table(describe_cols(G, list(shp), shp), "logo_shape_summary.csv", "Shape and stroke", "Shape, contour and stroke metrics.")
sym = {"sym_h": "Left-right mirror symmetry (IoU)", "sym_v": "Top-bottom mirror symmetry (IoU)", "sym_point": "180-degree point symmetry (IoU)", "centroid_x_offset": "Centroid horizontal offset (fraction of width)",
       "centroid_y_offset": "Centroid vertical offset (fraction of height)", "weight_tl": "Weight top-left", "weight_tr": "Weight top-right", "weight_bl": "Weight bottom-left", "weight_br": "Weight bottom-right"}
sd = describe_cols(G, list(sym), sym)
sd["Share >= symmetry threshold (%)"] = [pct((G[c] >= SYMMETRY_HIGH).sum(), G[c].notna().sum()) if c in ("sym_h", "sym_v", "sym_point") else np.nan for c in sym if c in G.columns]
save_table(sd, "logo_symmetry_balance.csv", "Shape and stroke", f"Symmetry (threshold IoU >= {SYMMETRY_HIGH}) and visual balance.")
for c, lab, fname in (("stroke_weight_class", "Relative stroke weight (dataset terciles)", "logo_stroke_weight_classes.csv"), ("angularity_class", "Contour character (dataset terciles of corner density)", "logo_angularity_classes.csv"),
                      ("complexity_class", "Complexity (dataset terciles of contour length)", "logo_complexity_classes.csv")):
    t = dist_table(G[c], lab) if c in G.columns else pd.DataFrame()
    R[c] = t; save_table(t, fname, "Shape and stroke" if c != "complexity_class" else "Scalability and complexity", f"{lab}. Classes are relative to this dataset.")

# ---- Scalability -------------------------------------------------------------------------------------------------------------
sc_rows = []
for T in SMALL_HEIGHTS:
    for c, lab in ((f"retention_iou_{T}", f"Shape retention at {T}px height (IoU)"), (f"component_retention_{T}", f"Component retention at {T}px height")):
        v = pd.to_numeric(G.get(c), errors="coerce").dropna()
        sc_rows.append({"Metric": lab, "Valid (n)": len(v), "Mean": v.mean() if len(v) else np.nan, "Median": v.median() if len(v) else np.nan, "P25": v.quantile(.25) if len(v) else np.nan, "Min": v.min() if len(v) else np.nan})
R["scal"] = save_table(pd.DataFrame(sc_rows), "logo_scalability.csv", "Scalability and complexity", "How much of the logo shape survives downscaling to small heights.")
G = G.assign(min_feature_px_24=G["stroke_weight_ratio"] * 24)
save_table(describe_cols(G, ["min_feature_px_24", "n_raw_components", "perimeter_norm", "corner_density"],
                         {"min_feature_px_24": "Estimated stroke width at 24px height (px)", "n_raw_components": "Connected components", "perimeter_norm": "Contour length / sqrt(area)", "corner_density": "Corners per 100 px of contour"}),
           "logo_complexity_metrics.csv", "Scalability and complexity", "Stroke width at small size and complexity metrics.")

# ---- Variants ----------------------------------------------------------------------------------------------------------------------
lg = a[a["role"].isin(["primary", "variant"]) & a["include_in_analysis"]]
var = lg.groupby("domain_key").agg(logos_analysed=("asset_id", "size"), distinct_color_modes=("color_mode", "nunique"), lightness_range=("fg_lightness", lambda s: s.max() - s.min()),
                                   has_svg=("is_svg", lambda s: bool(s.fillna(False).any()))).reset_index()
var["has_light_and_dark_versions"] = var["lightness_range"] >= 40
vs = pd.DataFrame([{"Metric": "Sites with more than one analysable logo version", "Sites (n)": int((var["logos_analysed"] > 1).sum()), "Base (n)": len(var)},
                   {"Metric": "Sites with versions differing in foreground lightness by >= 40 L*", "Sites (n)": int(var["has_light_and_dark_versions"].sum()), "Base (n)": len(var)},
                   {"Metric": "Sites with at least one SVG logo version", "Sites (n)": int(var["has_svg"].sum()), "Base (n)": len(var)},
                   {"Metric": "Sites with versions in different color modes", "Sites (n)": int((var["distinct_color_modes"] > 1).sum()), "Base (n)": len(var)}])
vs["Sites (%)"] = [pct(c, b) for c, b in zip(vs["Sites (n)"], vs["Base (n)"])]
save_table(vs, "logo_variants_summary.csv", "Variants", "Availability of alternative logo versions per site.")

,Metric,Sites (n),Base (n),Sites (%)
0,Sites with more than one analysable logo version,31,71,43.66
1,Sites with versions differing in foreground li...,16,71,22.54
2,Sites with at least one SVG logo version,0,71,0.00
3,Sites with versions in different color modes,15,71,21.13


## 8. SVG structure, icons and archetype clustering

In [10]:
# ---- SVG ------------------------------------------------------------------------------------------------------------------------------
svgs = a[a["is_svg"].fillna(False).astype(bool) & a["role"].isin(["primary", "variant"])]
if len(svgs):
    n = len(svgs)
    srows = [{"Property": lab, "Logos (n)": int((svgs[c] > 0).sum()), "Logos (%)": pct((svgs[c] > 0).sum(), n), "Base SVG logos (n)": n}
             for lab, c in (("Contains outlined paths", "svg_paths"), ("Contains basic shapes (circle, rect...)", "svg_basic_shapes"), ("Contains live text elements", "svg_text_elements"),
                            ("Contains gradients", "svg_gradients"), ("Uses clip paths or masks", "svg_clip_or_mask"), ("Embeds a raster image", "svg_embedded_raster"),
                            ("Depends on CSS classes or style blocks", "svg_uses_css_classes"), ("Uses currentColor", "svg_uses_currentcolor"), ("Declares a viewBox", "svg_has_viewbox"))]
    R["svg"] = save_table(pd.DataFrame(srows), "svg_structure_summary.csv", "SVG structure", "Structural properties of vector logos.")
    save_table(describe_cols(svgs, ["svg_bytes", "svg_paths", "svg_path_data_chars", "svg_distinct_fills", "svg_viewbox_aspect"],
                             {"svg_bytes": "SVG file size (bytes)", "svg_paths": "Path elements", "svg_path_data_chars": "Path data length (characters)", "svg_distinct_fills": "Distinct fill/stroke colors declared", "svg_viewbox_aspect": "viewBox aspect ratio (W/H)"}),
               "svg_structure_metrics.csv", "SVG structure", "Size and complexity of SVG logos.")
else:
    validate("SVG structure", "SVG logos", "NOT_GENERATED", "No SVG logo assets analysed.")

# ---- Icons --------------------------------------------------------------------------------------------------------------------------------
icons = a[a["role"].isin(["favicon", "apple_touch_icon"]) & a["analysis_status"].eq("ok")].copy()


def square_pad(m):
    h, w = m.shape; s = max(h, w); out = np.zeros((s, s), np.uint8); out[(s - h) // 2:(s - h) // 2 + h, (s - w) // 2:(s - w) // 2 + w] = m; return out


def dhash(m, size=8):
    r = cv2.resize(square_pad(m), (size + 1, size), interpolation=cv2.INTER_AREA)
    return (r[:, 1:] > r[:, :-1]).ravel()


def symbol_mask(asset_id):
    p = PROC.get(asset_id)
    if not p:
        return None
    sym = [g for g in p["groups"] if not g["text_like"] and g["share"] >= MIN_GROUP_SHARE]
    if not sym:
        return None
    g = max(sym, key=lambda g: g["area"])
    return p["mask"][g["y"]:g["y"] + g["h"], g["x"]:g["x"] + g["w"]]


if len(icons):
    icons["icon_form"] = np.select([(icons["canvas_fg_ratio"] >= 0.8) & (icons["corners_transparent"] >= 0.75), icons["canvas_fg_ratio"] >= 0.8, icons["has_alpha"].fillna(False).astype(bool)],
                                   ["Filled tile with rounded / cut corners", "Filled square tile", "Transparent glyph (no tile)"], default="Opaque with background")
    rep_ids = rep.set_index("domain_key")["asset_id"].to_dict()
    de, hs, hsym = [], [], []
    rep_lab = rep.set_index("domain_key")[["dominant_hex"]]
    for r in icons.itertuples():
        pid = rep_ids.get(r.domain_key)
        de.append(np.nan); hs.append(np.nan); hsym.append(np.nan)
        if pid is None or pid not in PROC or r.asset_id not in PROC:
            continue
        la = to_lab(np.array([[int(r.dominant_hex[5:7], 16), int(r.dominant_hex[3:5], 16), int(r.dominant_hex[1:3], 16)]]))[0]
        lp = rep.loc[rep["asset_id"] == pid, "dominant_hex"].iloc[0]
        lb = to_lab(np.array([[int(lp[5:7], 16), int(lp[3:5], 16), int(lp[1:3], 16)]]))[0]
        de[-1] = float(np.linalg.norm(la - lb))
        h_icon = dhash(PROC[r.asset_id]["mask"])
        hs[-1] = float((h_icon != dhash(PROC[pid]["mask"])).mean())
        sm = symbol_mask(pid)
        if sm is not None and sm.size > 64:
            hsym[-1] = float((h_icon != dhash(sm)).mean())
    icons["color_deltaE_vs_logo"], icons["shape_hash_distance_vs_logo"], icons["shape_hash_distance_vs_symbol"] = de, hs, hsym
    icons["best_shape_distance"] = icons[["shape_hash_distance_vs_logo", "shape_hash_distance_vs_symbol"]].min(axis=1)
    R["icons"] = icons
    for typ, lab in (("favicon", "Favicon"), ("apple_touch_icon", "Apple touch icon")):
        sub = icons[icons["role"] == typ]
        if len(sub):
            t = dist_table(sub["icon_form"], "Icon form"); t.insert(0, "Icon type", lab)
            R["icon_form_" + typ] = t
    save_table(pd.concat([R[k] for k in R if str(k).startswith("icon_form_")]) if any(str(k).startswith("icon_form_") for k in R) else None, "icons_form.csv", "Icons", "Icon form: tile, rounded tile or transparent glyph.")
    ic_lab = {"orig_w": "Icon width (px)", "orig_h": "Icon height (px)", "color_deltaE_vs_logo": "Dominant color distance to logo (deltaE)", "shape_hash_distance_vs_logo": "Shape hash distance to full logo (0-1)",
              "shape_hash_distance_vs_symbol": "Shape hash distance to logo symbol (0-1)", "best_shape_distance": "Best shape distance (full logo or symbol)"}
    save_table(describe_cols(icons, list(ic_lab), ic_lab), "icons_logo_similarity.csv", "Icons", "Icon resolution and similarity to the representative logo (lower distance = more similar; hash is a coarse measure).")
else:
    validate("Icons", "Icon analysis", "NOT_GENERATED", "No analysable favicons or apple touch icons.")

# ---- Archetype clustering ------------------------------------------------------------------------------------------------------------------------
C = BASIS.copy()
C["log_content_aspect"] = np.log(C["content_aspect_ratio"])
CF = {"log_content_aspect": "Log aspect ratio", "fg_fill_ratio": "Fill ratio", "n_colors": "Effective colors", "core_color_spread": "Color spread", "circularity_main": "Circularity",
      "global_solidity": "Global solidity", "corner_density": "Corner density", "sym_h": "Mirror symmetry", "stroke_weight_ratio": "Stroke weight", "perimeter_norm": "Contour length",
      "n_groups": "Visual groups", "symbol_area_share": "Symbol area share", "retention_iou_24": "Retention at 24px"}
usable = [c for c in CF if c in C.columns and C[c].notna().mean() >= 0.7]
R["cluster_ok"] = False
if len(C) >= 10 and len(usable) >= 4:
    X = C[usable].apply(pd.to_numeric, errors="coerce")
    X = X.fillna(X.median())
    Z = ((X - X.mean()) / X.std().replace(0, 1)).fillna(0).values.astype(np.float32)
    D = np.linalg.norm(Z[:, None] - Z[None], axis=2)

    def silhouette(labels):
        s = []
        for i in range(len(Z)):
            same = labels == labels[i]
            if same.sum() < 2:
                s.append(0); continue
            ai = D[i, same].sum() / (same.sum() - 1)
            bi = min(D[i, labels == c].mean() for c in set(labels) if c != labels[i])
            s.append((bi - ai) / max(ai, bi))
        return float(np.mean(s))

    sel, best = [], None
    for k in [k for k in CLUSTER_K_RANGE if k < len(C)]:
        _, lab_k, _ = cv2.kmeans(Z, k, None, (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 100, 1e-3), 10, cv2.KMEANS_PP_CENTERS)
        lab_k = lab_k.ravel(); s = silhouette(lab_k) if len(set(lab_k)) > 1 else np.nan
        sel.append({"Clusters (k)": k, "Mean silhouette": s})
        if best is None or (pd.notna(s) and s > best[0]):
            best = (s, k, lab_k)
    R["cluster_sel"] = save_table(pd.DataFrame(sel), "archetype_cluster_selection.csv", "Archetypes", "Silhouette score by number of clusters (the highest is used).")
    C["archetype"] = ["Archetype %d" % (i + 1) for i in best[2]]
    U_, S_, Vt = np.linalg.svd(Z - Z.mean(0), full_matrices=False)
    C["pc1"], C["pc2"] = U_[:, 0] * S_[0], U_[:, 1] * S_[1]
    R["pca_var"] = (S_ ** 2 / (S_ ** 2).sum())[:2]
    prof = C.groupby("archetype").agg(**{"Logos (n)": ("brand_key", "size")}).reset_index()
    for c in usable:
        prof[CF[c] + " (median)"] = prof["archetype"].map(C.groupby("archetype")[c].median())
    prof["Most common composition type"] = prof["archetype"].map(C.groupby("archetype")["composition_type"].agg(lambda s: s.mode().iloc[0] if len(s.mode()) else ""))
    prof["Most common color mode"] = prof["archetype"].map(C.groupby("archetype")["color_mode"].agg(lambda s: s.mode().iloc[0] if len(s.mode()) else ""))
    prof["Share of logos (%)"] = [pct(v, len(C)) for v in prof["Logos (n)"]]
    save_table(prof.rename(columns={"archetype": "Archetype"}), "archetype_profiles.csv", "Archetypes", "Median feature profile of each data-driven archetype (labels are numeric; interpretation is left to the analyst).")
    save_table(C[["archetype", "brand_key", "domain_key", "composition_type", "color_mode"]].rename(columns={"archetype": "Archetype", "brand_key": "Brand", "domain_key": "Domain", "composition_type": "Composition type", "color_mode": "Color mode"}).sort_values("Archetype"),
               "archetype_members.csv", "Archetypes", "Cluster membership of each logo.")
    R["cluster_ok"], R["C"] = True, C
else:
    validate("Archetypes", "Clustering", "NOT_GENERATED", f"Needs >= 10 logos and >= 4 features with 70% coverage (have {len(C)} logos, {len(usable)} features).")
R["G"] = G

## 9. Charts

In [11]:
nl = f"Analysis basis: {BASIS_LABEL} (n = {len(G)})."

# 01 Inventory
fig, ax = new_fig(2, 2, "Logo Collection Inventory")
p_bar(ax[0], st["Download status"], st["Logos (%)"], "Download status of asset rows", "Asset rows (%)", n=N_ROWS, xmax=100)
at = meta.groupby("asset_type").size().reset_index(name="n"); at["p"] = at["n"] / at["n"].sum() * 100
p_bar(ax[1], at["asset_type"], at["p"], "Detected asset types", "Asset rows (%)", n=N_ROWS, xmax=100)
p_bar(ax[2], fmt["Detected format"], fmt["Assets (%)"], "Formats of downloaded assets", "Downloaded assets (%)", n=int(fmt["Base (n)"].iloc[0]), xmax=100)
p_hist(ax[3], per_site["logo_rows"], "Logo candidates detected per site", "Logo candidates", bins=max(int(per_site["logo_rows"].max()), 1) + 1)
save_fig(fig, "01_inventory.png", "Inventory", "Download outcomes, asset types, formats and candidates per site.", f"Asset rows: n = {N_ROWS}.")

# 02 Geometry
fig, ax = new_fig(2, 2, "Logo Geometry")
p_hist(ax[0], G["content_aspect_ratio"], "Aspect ratio of the logo (tight crop)", "Width / height (log scale)", bins=25, log_x=True)
o = R["orientation"]; p_bar(ax[1], o["Aspect class"], o["Logos (%)"], "Orientation classes", "Logos (%)", n=int(o["Base (n)"].iloc[0]), xmax=100)
p_hist(ax[2], G["content_area_ratio"], "Content area / original canvas area", "Share of canvas used", bins=20)
p_hist(ax[3], G["fg_fill_ratio"], "Foreground fill of the bounding box", "Foreground pixels / bounding-box pixels", bins=20)
save_fig(fig, "02_geometry.png", "Geometry", "Aspect ratio, orientation, canvas usage and fill.", nl)

# 03 Composition
fig, ax = new_fig(2, 2, "Logo Composition")
c = R["comp"]; p_bar(ax[0], c["Composition type"], c["Logos (%)"], "Composition type (heuristic)", "Logos (%)", n=int(c["Base (n)"].iloc[0]), xmax=100)
l = R["layout"]; p_bar(ax[1], l["Symbol position (combination logos)"], l["Logos (%)"], "Symbol position (combination logos)", "Combination logos (%)", n=int(l["Base (n)"].iloc[0]) if len(l) else 0, xmax=100)
p_hist(ax[2], G["symbol_to_text_height_ratio"], "Symbol height / wordmark height", "Height ratio", bins=15)
ct = pd.crosstab(G["composition_type"], G["aspect_class"].astype(object)); share = ct.div(ct.sum(axis=1), axis=0) * 100
p_stack(ax[3], share, "Orientation within each composition type", "Share of logos (%)")
save_fig(fig, "03_composition.png", "Composition", "Composition types, symbol position, symbol/text proportion and orientation.", nl)

# 04 Color
fig, ax = new_fig(2, 3, "Logo Color", figsize=(21, 10.5))
m = R["cmode"]; p_bar(ax[0], m["Color mode"], m["Logos (%)"], "Color mode", "Logos (%)", n=int(m["Base (n)"].iloc[0]), xmax=100)
p_hist(ax[1], G["n_colors"], "Effective colors per logo", "Colors", bins=np.arange(.5, 8.5, 1))
f_ = R["cfam"]; p_bar(ax[2], f_["Dominant color family"], f_["Logos (%)"], "Dominant color family", "Logos (%)", n=int(f_["Base (n)"].iloc[0]), colors=[FAMILY_COLORS.get(x, C_GREY) for x in f_["Dominant color family"]], xmax=100)
bg = dist_table(G["bg_type"], "Background"); p_bar(ax[3], bg["Background"], bg["Logos (%)"], "Original background type", "Logos (%)", n=int(bg["Base (n)"].iloc[0]), xmax=100)
ld = R["lightness"]; p_bar(ax[4], ld["Foreground lightness class"], ld["Logos (%)"], "Foreground lightness", "Logos (%)", n=int(ld["Base (n)"].iloc[0]), xmax=100)
p_hist(ax[5], G["contrast_on_white"], "Dominant color contrast on white", "WCAG contrast ratio", bins=20)
save_fig(fig, "04_color.png", "Color", "Color mode, palette size, dominant family, background, lightness and contrast.", nl)

# 05 Shape and stroke
fig, ax = new_fig(2, 3, "Shape, Stroke and Symmetry", figsize=(21, 10.5))
p_hist(ax[0], G["circularity_main"], "Circularity of the largest shape", "Circularity (1 = circle)", bins=20)
p_hist(ax[1], G["global_solidity"], "Global solidity", "Foreground / convex hull area", bins=20)
p_hist(ax[2], G["corner_density"], "Corner density", "Corners per 100 px of contour", bins=20)
p_hist(ax[3], G["stroke_weight_ratio"], "Relative stroke weight", "Stroke width / logo height", bins=20)
p_hist(ax[4], G["sym_h"], "Left-right mirror symmetry", "IoU with mirrored logo", bins=20)
pp = G[["centroid_x_offset", "centroid_y_offset"]].dropna()
if len(pp):
    ax[5].scatter(pp["centroid_x_offset"], pp["centroid_y_offset"], alpha=.65, color=C_PRIMARY, edgecolor="white"); ax[5].axhline(0, color=C_GREY, lw=.8); ax[5].axvline(0, color=C_GREY, lw=.8)
    ax[5].invert_yaxis(); ax[5].set_xlabel("Horizontal offset (fraction of width)"); ax[5].set_ylabel("Vertical offset (fraction of height)"); ax[5].set_title(f"Visual centre of mass (n = {len(pp)})", loc="left")
else:
    empty_panel(ax[5], "Visual centre of mass")
save_fig(fig, "05_shape_and_stroke.png", "Shape and stroke", "Circularity, solidity, corners, stroke weight, symmetry and balance.", nl)

# 06 Scalability
fig, ax = new_fig(2, 2, "Scalability and Complexity")
data = [(f"{T}px", pd.to_numeric(G[f"retention_iou_{T}"], errors="coerce").dropna()) for T in SMALL_HEIGHTS if f"retention_iou_{T}" in G.columns]
data = [d for d in data if len(d[1])]
if data:
    ax[0].boxplot([d[1] for d in data], patch_artist=True, boxprops=dict(facecolor="#BBD3E8"), medianprops=dict(color=C_ACCENT, lw=2))
    ax[0].set_xticks(range(1, len(data) + 1)); ax[0].set_xticklabels([d[0] for d in data])
    ax[0].set_ylabel("Shape retention (IoU)"); ax[0].set_xlabel("Rendered logo height"); ax[0].set_title("Shape retention at small sizes", loc="left")
else:
    empty_panel(ax[0], "Shape retention at small sizes")
pp = G[["stroke_weight_ratio", "retention_iou_24"]].dropna()
if len(pp):
    ax[1].scatter(pp["stroke_weight_ratio"], pp["retention_iou_24"], alpha=.65, color=C_PRIMARY, edgecolor="white"); ax[1].set_xlabel("Stroke width / logo height"); ax[1].set_ylabel("Retention at 24px (IoU)"); ax[1].set_title(f"Stroke weight vs small-size retention (n = {len(pp)})", loc="left")
else:
    empty_panel(ax[1], "Stroke weight vs small-size retention")
p_hist(ax[2], G["perimeter_norm"], "Contour length / sqrt(area)", "Complexity index", bins=20)
p_hist(ax[3], G["n_raw_components"], "Connected components per logo", "Components", bins=20)
save_fig(fig, "06_scalability_and_complexity.png", "Scalability and complexity", "Small-size retention, stroke vs retention, contour complexity and components.", nl)

# 07 Icons
fig, ax = new_fig(2, 2, "Favicons and App Icons")
ic = R.get("icons")
if ic is not None and len(ic):
    form = pd.crosstab(ic["role"], ic["icon_form"]); share = form.div(form.sum(axis=1), axis=0) * 100
    p_stack(ax[0], share, "Icon form", "Share of icons (%)", n_text=f" (n = {len(ic)})")
    p_hist(ax[1], ic["orig_w"], "Icon width", "Pixels (log scale)", bins=15, log_x=True)
    p_hist(ax[2], ic["color_deltaE_vs_logo"], "Dominant color distance to the logo", "deltaE (CIE76)", bins=20)
    p_hist(ax[3], ic["best_shape_distance"], "Shape hash distance to logo / symbol", "Normalised Hamming distance (0 = identical)", bins=20)
else:
    for x in ax: empty_panel(x, "Icons", "No icon data available")
save_fig(fig, "07_icons.png", "Icons", "Icon form, resolution and similarity to the logo.", f"Icon assets analysed: n = {0 if ic is None else len(ic)}.")

# 08 SVG
fig, ax = new_fig(1, 3, "Vector (SVG) Logo Structure", figsize=(21, 6))
sv = R.get("svg")
if sv is not None:
    p_bar(ax[0], sv["Property"], sv["Logos (%)"], "SVG structural properties", "SVG logos (%)", n=int(sv["Base SVG logos (n)"].iloc[0]), xmax=100)
    p_hist(ax[1], svgs["svg_paths"], "Path elements per SVG logo", "Paths", bins=20)
    p_hist(ax[2], svgs["svg_distinct_fills"], "Distinct declared colors per SVG logo", "Colors", bins=np.arange(-.5, 10.5, 1))
else:
    for x in ax: empty_panel(x, "SVG structure", "No SVG logos analysed")
save_fig(fig, "08_svg_structure.png", "SVG structure", "Structural properties of SVG logos.", f"SVG logos: n = {len(svgs)}.")

# 09 Archetypes
fig, ax = new_fig(1, 3, "Data-driven Logo Archetypes", figsize=(21, 6.5))
if R["cluster_ok"]:
    Cc = R["C"]
    for i, (name, grp) in enumerate(Cc.groupby("archetype")):
        ax[0].scatter(grp["pc1"], grp["pc2"], color=CAT[i % len(CAT)], label=f"{name} (n = {len(grp)})", alpha=.8, edgecolor="white", s=55)
    ax[0].set_xlabel(f"PC1 ({R['pca_var'][0] * 100:.0f}% variance)"); ax[0].set_ylabel(f"PC2 ({R['pca_var'][1] * 100:.0f}% variance)"); ax[0].set_title("Logo feature space (PCA)", loc="left"); ax[0].legend()
    sizes = Cc["archetype"].value_counts().sort_index(); p_bar(ax[1], sizes.index, sizes.values / len(Cc) * 100, "Archetype sizes", "Logos (%)", n=len(Cc), xmax=100)
    cs = R["cluster_sel"]; ax[2].plot(cs["Clusters (k)"], cs["Mean silhouette"], marker="o", color=C_PRIMARY); ax[2].set_xlabel("Number of clusters (k)"); ax[2].set_ylabel("Mean silhouette"); ax[2].set_title("Cluster selection", loc="left")
else:
    for x in ax: empty_panel(x, "Archetypes", "Not enough logos or features for clustering")
save_fig(fig, "09_archetypes.png", "Archetypes", "PCA map, archetype sizes and silhouette-based selection.", nl)

## 10. Contact sheets and validation overlays
The overlay sheet shows the detected bounding groups (blue = text-like, red = symbol-like). **Use it to verify the composition heuristic**
and tune `GROUP_GAP_RATIO`, `TEXT_MIN_COMPONENTS`, `TEXT_MIN_ASPECT` and `WORDMARK_ASPECT` if needed.

In [12]:
BG_DISPLAY = 190


def display_image(p, overlay=False):
    bgr, mask = p["bgr"], p["mask"]
    al = (mask.astype(np.float32) / 255)[..., None]
    img = (bgr * al + BG_DISPLAY * (1 - al)).astype(np.uint8)
    pad = max(6, img.shape[0] // 12)
    img = cv2.copyMakeBorder(img, pad, pad, pad, pad, cv2.BORDER_CONSTANT, value=(BG_DISPLAY,) * 3)
    if overlay:
        t = max(2, img.shape[0] // 70)
        for g in p["groups"]:
            if g["share"] < MIN_GROUP_SHARE:
                continue
            color = (200, 100, 0) if g["text_like"] else (0, 0, 220)
            cv2.rectangle(img, (g["x"] + pad, g["y"] + pad), (g["x"] + g["w"] + pad, g["y"] + g["h"] + pad), color, t)
    return img


def make_sheet(items, filename, dim, desc, cols=6, cell=(300, 150)):
    if not items:
        validate(dim, filename, "NOT_GENERATED", "No logos to draw."); return
    cw, ch = cell; lab_h = 18; rows = math.ceil(len(items) / cols)
    sheet = np.full((rows * (ch + lab_h), cols * cw, 3), 235, np.uint8)
    for i, (label, img) in enumerate(items):
        r, c = divmod(i, cols)
        s = min((cw - 12) / img.shape[1], (ch - 12) / img.shape[0])
        im = cv2.resize(img, (max(1, int(img.shape[1] * s)), max(1, int(img.shape[0] * s))), interpolation=cv2.INTER_AREA)
        y0, x0 = r * (ch + lab_h) + (ch - im.shape[0]) // 2, c * cw + (cw - im.shape[1]) // 2
        sheet[y0:y0 + im.shape[0], x0:x0 + im.shape[1]] = im
        cv2.putText(sheet, str(label)[:38], (c * cw + 6, r * (ch + lab_h) + ch + 13), cv2.FONT_HERSHEY_SIMPLEX, 0.38, (40, 40, 40), 1, cv2.LINE_AA)
    cv2.imencode(".png", sheet)[1].tofile(str(SHEETS_DIR / filename))
    MANIFEST.append({"Dimension": dim, "Type": "contact sheet", "Filename": f"contact_sheets/{filename}", "Rows": len(items), "Description": desc})


order = BASIS.sort_values(["composition_type", "aspect_class", "brand_key"])
make_sheet([(f"{r.brand_key} | {r.composition_type[:12]}", display_image(PROC[r.asset_id])) for r in order.itertuples() if r.asset_id in PROC],
           "contact_sheet_representative_logos.png", "Composition", "Representative logos ordered by composition type and orientation.")
make_sheet([(f"{r.brand_key} | {r.composition_type[:14]}", display_image(PROC[r.asset_id], overlay=True)) for r in order.itertuples() if r.asset_id in PROC],
           "contact_sheet_composition_overlays.png", "Composition", "Detected visual groups (blue = text-like, red = symbol-like) to validate the composition heuristic.", cols=4, cell=(420, 190))
if R["cluster_ok"]:
    for name, grp in R["C"].groupby("archetype"):
        make_sheet([(r.brand_key, display_image(PROC[r.asset_id])) for r in grp.itertuples() if r.asset_id in PROC],
                   f"contact_sheet_{name.lower().replace(' ', '_')}.png", "Archetypes", f"Logos assigned to {name}.", cols=5)
if len(icons):
    make_sheet([(f"{r.brand_key} | {r.role[:8]}", display_image(PROC[r.asset_id])) for r in icons.sort_values(["role", "brand_key"]).itertuples() if r.asset_id in PROC],
               "contact_sheet_icons.png", "Icons", "Favicons and apple touch icons.", cols=10, cell=(150, 110))

## 11. Validation report, Excel workbook and manifest

In [13]:
for dim, name in (("Inventory", "inventory"), ("Composition", "composition"), ("Color", "color")):
    pass
if not SVG_RENDER_OK:
    validate("SVG structure", "SVG rendering", "UNAVAILABLE", "cairosvg is not installed (pip install cairosvg): SVG logos are not rasterised, so OpenCV features are skipped for them; SVG structure is still analysed.")
validate("Composition", "Composition classification", "INFO", "Heuristic based on connected components and spatial grouping; check contact_sheet_composition_overlays.png and provide manual labels.", len(BASIS))
validate("Color", "Palette extraction", "INFO", "k-means in Lab on eroded foreground pixels, similar clusters merged; colors below the share threshold are dropped.", len(BASIS))
val = pd.DataFrame(VALIDATION)
val.to_csv(TABLES / "validation_report.csv", index=False, encoding="utf-8-sig")
MANIFEST.append({"Dimension": "Validation", "Type": "table", "Filename": "tables/validation_report.csv", "Rows": len(val), "Description": "Column checks, exclusions and heuristic limitations."})
EXCEL_TABLES.setdefault("Validation", []).append(("validation_report", "Column checks, exclusions and heuristic limitations.", val))

try:
    from openpyxl.styles import Font
    with pd.ExcelWriter(OUT / "logo_report_data.xlsx", engine="openpyxl") as writer:
        for dim in DIM:
            tabs = EXCEL_TABLES.get(dim, [])
            if not tabs:
                continue
            sheet = dim[:31]; row = 0
            for name, desc, df in tabs:
                df.to_excel(writer, sheet_name=sheet, startrow=row + 1, index=False)
                writer.sheets[sheet].cell(row=row + 1, column=1, value=f"{name}: {desc}").font = Font(bold=True)
                row += len(df) + 4
            for col in writer.sheets[sheet].columns:
                writer.sheets[sheet].column_dimensions[col[0].column_letter].width = min(max(max((len(str(c.value)) for c in col if c.value is not None), default=10) + 2, 12), 60)
        pd.DataFrame(MANIFEST).to_excel(writer, sheet_name="Manifest", index=False)
    MANIFEST.append({"Dimension": "All", "Type": "workbook", "Filename": "logo_report_data.xlsx", "Rows": np.nan, "Description": "All tables, one sheet per dimension."})
except ImportError:
    validate("Validation", "Excel workbook", "NOT_GENERATED", "openpyxl not installed.")

pd.DataFrame(MANIFEST).to_csv(OUT / "logo_report_manifest.csv", index=False, encoding="utf-8-sig")
print("=" * 60)
print("LOGO ANALYSIS COMPLETED")
print(f"Analysis basis      : {BASIS_LABEL} (n = {len(BASIS)})")
print(f"Tables              : {sum(m['Type'] == 'table' for m in MANIFEST)}")
print(f"Charts              : {sum(m['Type'] == 'chart' for m in MANIFEST)}")
print(f"Contact sheets      : {sum(m['Type'] == 'contact sheet' for m in MANIFEST)}")
print(f"Validation findings : {len(val)} (tables/validation_report.csv)")
print(f"Output directory    : {OUT.resolve()}")
print("=" * 60)

LOGO ANALYSIS COMPLETED
Analysis basis      : one logo per site (n = 71)
Tables              : 47
Charts              : 9
Contact sheets      : 5
Validation findings : 24 (tables/validation_report.csv)
Output directory    : D:\chapa.com\analisis-front\analysis\logos
